# Cosmological Perturbation Theory: $\delta R_{\mu\nu}$

**Background**: $\bar{g}_{\mu\nu} = \mathrm{diag}(-N^2,\; a^2\delta_{ij})$, $\quad k=0$

**Scalar perturbation (Newtonian gauge, $B=E=0$)**:
$$\delta g_{00} = -2N^2 A, \qquad \delta g_{ij} = 2a^2 C\,\delta_{ij}$$

In [1]:
import sys; sys.path.insert(0, "/home/minjae/Minjae/IndexCalc")
import sympy as sp
from IPython.display import display, Math

t, x, y, z = sp.symbols('t x y z')
eps = sp.Symbol('eps')
spatial = [x, y, z]
coords = [t, x, y, z]
dim = 4

a = sp.Function('a', positive=True)(t)
N = sp.Function('N', positive=True)(t)
A = sp.Function('A')(t, x, y, z)
C = sp.Function('C')(t, x, y, z)

def linearize(expr):
    return expr.series(eps, 0, 2).removeO()

# Metric
g = [[sp.S(0)]*dim for _ in range(dim)]
g[0][0] = -N**2 * (1 + 2*eps*A)
for i in range(1, 4):
    g[i][i] = a**2 * (1 + 2*eps*C)

g_inv = [[sp.S(0)]*dim for _ in range(dim)]
g_inv[0][0] = -1/N**2 * (1 - 2*eps*A)
for i in range(1, 4):
    g_inv[i][i] = 1/a**2 * (1 - 2*eps*C)

# Christoffel
Gamma = [[[None]*dim for _ in range(dim)] for _ in range(dim)]
for s in range(dim):
    for m in range(dim):
        for n in range(m, dim):
            val = sp.S(0)
            for rho in range(dim):
                if g_inv[s][rho] == 0: continue
                val += g_inv[s][rho] * (
                    sp.diff(g[rho][m], coords[n])
                    + sp.diff(g[rho][n], coords[m])
                    - sp.diff(g[m][n], coords[rho]))
            val = linearize(sp.expand(sp.Rational(1,2)*val))
            Gamma[s][m][n] = val
            Gamma[s][n][m] = val

bg  = [[[G.coeff(eps,0) for G in row] for row in mat] for mat in Gamma]
dG  = [[[G.coeff(eps,1) for G in row] for row in mat] for mat in Gamma]

def dR(mu, nu):
    val = sp.S(0)
    for rho in range(dim):
        val += sp.diff(dG[rho][mu][nu], coords[rho])
        val -= sp.diff(dG[rho][mu][rho], coords[nu])
        for sigma in range(dim):
            val += bg[rho][rho][sigma]*dG[sigma][mu][nu]
            val += dG[rho][rho][sigma]*bg[sigma][mu][nu]
            val -= bg[rho][mu][sigma]*dG[sigma][nu][rho]
            val -= dG[rho][mu][sigma]*bg[sigma][nu][rho]
    return sp.expand(val)

results = {}
for mu in range(dim):
    for nu in range(mu, dim):
        results[(mu,nu)] = dR(mu, nu)

print("Computation done.")

Computation done.


## LaTeX 표기 변환

$\dot{A} \to \dot{A}$, $\nabla^2 A \to \nabla^2 A$, $H = \dot{a}/a$ 등으로 정리.

In [2]:
# ── LaTeX 표기 변환 함수 ──

# 기호 정의
Adot  = sp.Symbol(r'\dot{A}')
Cdot  = sp.Symbol(r'\dot{C}')
Addot = sp.Symbol(r'\ddot{A}')
Cddot = sp.Symbol(r'\ddot{C}')
lap_A = sp.Symbol(r'\nabla^2 A')
lap_C = sp.Symbol(r'\nabla^2 C')
A_s   = sp.Symbol('A')
C_s   = sp.Symbol('C')
H_s   = sp.Symbol('H')         # ȧ/a
Hdot  = sp.Symbol(r'\dot{H}')  # d/dt(ȧ/a)
N_s   = sp.Symbol('N')
Ndot  = sp.Symbol(r'\dot{N}')
a_s   = sp.Symbol('a')

# 치환 규칙: SymPy Function 표현 → 깔끔한 기호
def prettify(expr, component="general"):
    """SymPy 결과를 깔끔한 LaTeX 기호로 치환."""
    subs = {
        sp.Derivative(A, t, 2): Addot,
        sp.Derivative(C, t, 2): Cddot,
        sp.Derivative(A, t): Adot,
        sp.Derivative(C, t): Cdot,
        sp.Derivative(N, t): Ndot,
        A: A_s,
        C: C_s,
        N: N_s,
        a: a_s,
    }
    e = expr.subs(subs)

    # spatial Laplacian: ∂²/∂x² + ∂²/∂y² + ∂²/∂z²
    # For (0,0) and scalar: full Laplacian
    A_func = sp.Function('A')(t, x, y, z)
    C_func = sp.Function('C')(t, x, y, z)

    # Replace spatial second derivatives with ∇²
    for f, lap_sym in [(A_func, lap_A), (C_func, lap_C)]:
        f_subs = subs.get(f, f)
        lap_val = sum(sp.diff(f, xi, 2) for xi in spatial)
        lap_val_sub = lap_val.subs(subs)
        # Check if expression contains the full Laplacian
        e_test = e.subs(lap_val_sub, lap_sym)
        if e_test != e:
            e = e_test

    # For R_{xx}: ∂²A/∂x² appears separately — handle trace-traceless split
    if component == "spatial_diag":
        # δR_{xx} has ∂_x² A + ∂_x² C separately, not full ∇²
        # Keep as is, but note: ∇²C appears, ∂_x²A does not combine
        pass

    return e


def show_latex(label, expr, component="general"):
    """LaTeX로 렌더링."""
    e = prettify(expr, component)
    latex_str = sp.latex(e)
    display(Math(rf"{label} = {latex_str}"))

print("Ready.")

Ready.


## $\delta R_{00}$

In [3]:
show_latex(r"\delta R_{00}", results[(0,0)])

<IPython.core.display.Math object>

## $\delta R_{0i}$

$\delta R_{0i}$는 $\partial_i(\cdots)$ 형태. 아래는 $i=x$ 성분:

In [4]:
# δR_{0i} = ∂_i × [coefficient]
# Factor out ∂/∂x from δR_{0x}
val_0i = results[(0,1)]
# Extract the coefficient: δR_{0x} = ∂_x(f) → f is the coefficient
# Manually: δR_{0x} = 2(ȧ/a)∂_x A - 2∂_x Ċ = 2∂_x(HA - Ċ)
show_latex(r"\delta R_{0x}", val_0i)

# Show factored form
coeff_0i = sp.expand(val_0i / sp.diff(A, x))  # heuristic
display(Math(r"\delta R_{0i} = 2\,\partial_i\!\left(\frac{\dot{a}}{a}\,A - \dot{C}\right)"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## $\delta R_{ij}$

Trace + traceless 분해: $\delta R_{ij} = [\text{trace}]\,\delta_{ij} - \partial_i\partial_j(A+C)$

In [5]:
# δR_{xx} (full)
show_latex(r"\delta R_{xx}", results[(1,1)])

# δR_{xy} (traceless part)
show_latex(r"\delta R_{xy}", results[(1,2)])

# Trace part: 1/3 Σ δR_{ii}
trace = sp.Rational(1,3) * (results[(1,1)] + results[(2,2)] + results[(3,3)])
trace = sp.expand(trace)
display(Math(r"\text{--- Trace part: } \tfrac{1}{3}\sum_i \delta R_{ii} \text{ ---}"))
show_latex(r"\tfrac{1}{3}\textstyle\sum_i \delta R_{ii}", trace)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## $\delta R$ (Ricci scalar)

$$\delta R = \bar{g}^{\mu\nu}\delta R_{\mu\nu} + \delta g^{\mu\nu}\bar{R}_{\mu\nu}$$

In [6]:
# Background Ricci
R_bg = {}
Gamma_bg = bg
for mu in range(dim):
    for nu in range(mu, dim):
        val = sp.S(0)
        for rho in range(dim):
            val += sp.diff(Gamma_bg[rho][mu][nu], coords[rho])
            val -= sp.diff(Gamma_bg[rho][mu][rho], coords[nu])
            for sigma in range(dim):
                val += Gamma_bg[rho][rho][sigma]*Gamma_bg[sigma][mu][nu]
                val -= Gamma_bg[rho][mu][sigma]*Gamma_bg[sigma][nu][rho]
        R_bg[(mu,nu)] = sp.expand(val)

display(Math(r"\text{Background:}"))
show_latex(r"\bar{R}_{00}", R_bg[(0,0)])
show_latex(r"\bar{R}_{ii}", R_bg[(1,1)])

# δR = ḡ^{μν}δR_{μν} + δg^{μν}R̄_{μν}
delta_R = sp.S(0)
delta_R += (-1/N**2) * results[(0,0)]
for i in range(1,4):
    delta_R += (1/a**2) * results[(i,i)]
delta_R += (2*A/N**2) * R_bg[(0,0)]
for i in range(1,4):
    delta_R += (-2*C/a**2) * R_bg[(i,i)]
delta_R = sp.expand(delta_R)

display(Math(r"\text{---}"))
show_latex(r"\delta R", delta_R)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## $N=1$ (cosmic time) 특수화

In [7]:
N1 = {N: 1, sp.Derivative(N, t): 0, sp.Derivative(N, t, 2): 0}

display(Math(r"\text{Setting } N=1:"))
for key, label in [((0,0), r"\delta R_{00}"),
                    ((0,1), r"\delta R_{0x}"),
                    ((1,1), r"\delta R_{xx}"),
                    ((1,2), r"\delta R_{xy}")]:
    val = sp.simplify(results[key].subs(N1))
    show_latex(label, val)

delta_R_N1 = sp.simplify(delta_R.subs(N1))
display(Math(r"\text{---}"))
show_latex(r"\delta R\big|_{N=1}", delta_R_N1)

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>